Notebook for 'basic' experiments:
- Extracting nodes and relations right away with a simple prompt
- Extracting nodes and then relations
- Few-shot with the better approach of the above

References:
- https://github.com/robert-mcdermott/ai-knowledge-graph/blob/main/src/knowledge_graph/prompts.py
- https://aclanthology.org/2024.kallm-1.8.pdf (hallucinations, penalty parameters for llm)

In [ ]:
import os
import json
from typing import List, Dict, TypedDict
from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_ollama import ChatOllama

LLM_MODEL_NAME="jaro-jm/llm:latest"
# LLM_MODEL_NAME="SpeakLeash/bielik-11b-v2.3-instruct:Q8_0"
os.environ["HSA_OVERRIDE_GFX_VERSION"]="11.0.0"

llm_text = ChatOllama(model=LLM_MODEL_NAME, temperature=0, reasoning=False, num_ctx=35000, num_predict=25000)

text = """Donald Trump od początku traktował konflikt w Ukrainie jako nieswoją wojnę. Nie miał cierpliwości ani zrozumienia dla ukraińskich racji. Błędnie zakładał też, że zdoła oderwać Rosję od sojuszu z Chinami. Tymczasem oba kraje mocno zacieśniły współpracę gospodarczą, a Chiny dostarczają dziś Rosjanom maszyny i technologie, na które monopol miał kiedyś Zachód. Moskwa i Pekin pamiętają czasy, gdy były podzielone. Putin i Xi są zgodni, że rozpad Związku Radzieckiego nie był koniecznością, tylko skutkiem błędów Kremla. Rosję i Chiny dzieli przepaść kulturalna, bo Rosjanie nie chcą być Chińczykami, ale to akurat ułatwia relacje. Współpraca z Europą grozi w oczach Putina zarażeniem rosyjskiego społeczeństwa „demokratycznym wirusem”, a w kontaktach z Chinami takiego ryzyka nie ma. Rosja może jednak wpaść w nieodwracalną zależność od Chin. Decydując się na inwazję na Ukrainę, Putin chciał przede wszystkim zasilić ludność Rosji dziesiątkami milionów Słowian, co było dla niego ważniejsze od zdobyczy terytorialnych. W tle są obawy przed chińską dominacją."""

## Relations right away

In [ ]:
combined_prompt = ChatPromptTemplate.from_messages([
    ("system", """You extract explicit relations from text.

Your task:
- Extract 5-8 most important, explicitly stated relations as atomic triples:
  subject -> relation -> object

Rules:
- Work strictly on the provided text. No inference.
- Each triple must contain:
  - subject: canonical name of the actor/entity/concept
  - relation: short, precise, atomic verb phrase (e.g. wspiera, powoduje, zależy_od)
  - object: canonical name of the target entity/concept
- Do not generate triples without a clear object.
- Do not duplicate identical triples.
- Direction matters.
- Normalize names (e.g. ChRL, Chiny -> Chiny).
- Return everything in Polish.

Output:
- Valid JSON only:
{{"triples":[{{"subject":"","relation":"","object":""}}]}}
- If no relations exist, return {{"triples":[]}}
"""),
    ("user", "Text:\n{text}")
])


llm_chain = combined_prompt | llm_text
output = llm_chain.invoke({"text": text}).content
print(repr(output))

In [ ]:
output_json = json.loads(output)
for i, t in enumerate(output_json["triples"], start=1):
    print(f"{i:02d}. {t['subject']} ──[{t['relation']}]──▶ {t['object']}")

## Nodes first, then relations

In [ ]:
extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", """You analyze text for the purpose of building a knowledge graph.
Your task is to extract ONLY 12-17 most relevant:
 - entities,
 - events,
 - concepts,
 that can function as nodes in the knowledge graph.

Specifically identify:
- persons (individual human actors),
- organizations (companies, institutions, movements),
- locations (geographical and administrative places),
- events (historical, social, political, technical, scientific, etc.),
- time expressions (dates, periods, time ranges),
- key thematic concepts (ideas, technologies, phenomena, theories).

**Rules:**
- Return only semantically meaningful elements, that can exist independently as nodes in a knowledge graph.
- Use canonical names where possible (e.g. “European Union” instead of abbreviations, if context allows).
- Do not duplicate items.
- Do not add category labels, keys, explanations or metadata.
- Return everything in Polish.
- Avoid returning multiple surface forms of the same entity or concept.
  If different names refer to the same real-world entity (e.g. „ChRL”, „Chiny”, „Chińska Republika Ludowa”),
  return only one canonical form, chosen based on common usage in context.

**Output format:**
- A single line of text.
- All extracted elements separated by commas.
- No JSON, no lists, no bullet points, no headings, no comments."""),
    ("user", """Text:
    Donald Trump nie ogranicza się do przebudowy Białego Domu. Zmienia też to, jak Amerykanie patrzą
na świat. Wenezuela liczy się dla niego bardziej niż Ukraina, bo to z Ameryki Łacińskiej
pochodzą największe kłopoty dzisiejszych Stanów: nielegalna imigracja i przemyt narkotyków.
Ważny jest również Pacyfik i rywalizacja z Chinami o hegemonię w tym regionie.
Trump uważa, że USA traciły na porządku, który powstał po zakończeniu zimnej wojny.
Wciąż trwa spór o to, czy przyjęcie ChRL do WTO było dobrą decyzją.
Jednocześnie Ameryka wiele zyskiwała, między innymi dzięki temu,
że dolar pozostawał walutą rezerwową świata."""),
    ("ai", """Donald Trump, Biały Dom, Stany Zjednoczone, Amerykanie, Wenezuela, Ukraina, Ameryka Łacińska,
nielegalna imigracja, przemyt narkotyków, Pacyfik, Chiny, hegemonia, zimna wojna,
Światowa Organizacja Handlu, dolar, waluta rezerwowa świata"""),
    ("user", "Text: {text}")
])

extraction_chain = extraction_prompt | llm_text
extracted_content_raw = extraction_chain.invoke({"text": text})
extracted_content = extracted_content_raw.content
print(extracted_content_raw)

nodes = [
    item.strip()
    for item in extracted_content.split(",")
    if item.strip()
]

seen = set()
nodes = [x for x in nodes if not (x in seen or seen.add(x))]
print(nodes)

In [ ]:
relations_prompt = ChatPromptTemplate.from_messages([
    ("system", """You extract semantic relations for the purpose of building a knowledge graph.

You are given:
- a source text,
- a predefined list of nodes.

Your task is to identify 12-15 explicitly stated, most important relations between the provided nodes, using provided text as context.

Rules:
- Only use nodes from the provided Nodes list.
- Each triple must have a subject, a concise relation, and an object.
- Relations must be short, precise, atomic, and semantically meaningful (1-3 words).
- Do not generate free-text or descriptive relations.
- Do not omit the object. If no valid object exists, do not create a triple.
- If multiple relations exist between the same nodes, return them as separate triples.
- Do not duplicate identical triples.
- Direction matters: subject -> relation -> object.

Output format:
- JSON only.
- Must strictly match: {{"triples":[{{"subject":"","relation":"","object":""}}]}}
- No explanations or comments.
- If no valid relations exist, return: {{"triples":[]}}"""),
    ("user",
     "Text:\n{text}\n\n"
     "Nodes:\n{nodes}"
     )
])

relations_chain = relations_prompt | llm_text
relations_output = relations_chain.invoke({
    "text": text,
    "nodes": ", ".join(nodes)
})

In [ ]:
relations_output_json = json.loads(relations_output.content)
print("Relations:")
for triple in relations_output_json['triples']:
    print(f"{triple['subject']} -> {triple['relation']} -> {triple['object']}")

## As in the article

In [ ]:
combined_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an advanced AI system specialized in knowledge extraction and knowledge graph generation.
Your expertise includes identifying consistent entity references and meaningful relationships in text.
CRITICAL INSTRUCTION: All relationships (predicates) MUST be no more than 3 words maximum. Ideally 1-2 words. This is a hard limit.
"""),
    ("user", """Your task: Read the text below and identify all Subject-Predicate-Object (S-P-O) relationships in each sentence. Then produce a single JSON array of objects, each representing one triple.

     Follow these rules carefully:

     - Entity Consistency: Use consistent names for entities throughout the document. For example, if "John Smith" is mentioned as "John", "Mr. Smith", and "John Smith" in different places, use a single consistent form (preferably the most complete one) in all triples.
    - Atomic Terms: Identify distinct key terms (e.g., objects, locations, organizations, acronyms, people, conditions, concepts, feelings). Avoid merging multiple ideas into one term (they should be as "atomistic" as possible).
- Unified References: Replace any pronouns (e.g., "he," "she," "it," "they," etc.) with the actual referenced entity, if identifiable.
- Pairwise Relationships: If multiple terms co-occur in the same sentence (or a short paragraph that makes them contextually related), create one triple for each pair that has a meaningful relationship.
- CRITICAL INSTRUCTION: Predicates MUST be 1-3 words maximum. Never more than 3 words. Keep them extremely concise.
- Ensure that all possible relationships are identified in the text and are captured in an S-P-O relation.
- Standardize terminology: If the same concept appears with slight variations (e.g., "artificial intelligence" and "AI"), use the most common or canonical form consistently.
- Make all the text of S-P-O text lower-case, even Names of people and places.
- If a person is mentioned by name, create a relation to their location, profession and what they are known for (invented, wrote, started, title, etc.) if known and if it fits the context of the informaiton.

Important Considerations:
- Aim for precision in entity naming - use specific forms that distinguish between similar but different entities
- Maximize connectedness by using identical entity names for the same concepts throughout the document
- Consider the entire context when identifying entity references
- ALL PREDICATES MUST BE 3 WORDS OR FEWER - this is a hard requirement
- All predicates should be in polish

Output Requirements:

- Do not include any text or commentary outside of the JSON.
- Return only the JSON array, with each triple as an object containing "subject", "predicate", and "object".
- Make sure the JSON is valid and properly formatted.

Example of the desired output structure:

[
    {{
        "subject": "Term A",
        "predicate": "relates to",  # Notice: only 2 words
        "object": "Term B"
    }},
    {{
        "subject": "Term C",
        "predicate": "uses",  # Notice: only 1 word
        "object": "Term D"
    }}
]

Important: Only output the JSON array (with the S-P-O objects) and nothing else

Text to analyze:\n{text}""")
])


llm_chain = combined_prompt | llm_text
output = llm_chain.invoke({"text": text}).content
output

In [ ]:
output_json = json.loads(output)
for i, t in enumerate(output_json, start=1):
    print(f"{i:02d}. {t['subject']} ──[{t['predicate']}]──▶ {t['object']}")